<a href="https://colab.research.google.com/github/qiaosungithub/Qwen-RL-LoRA/blob/sft/sft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U torch==2.8.0 torchvision transformers datasets accelerate trl evaluate bitsandbytes wandb peft
!pip install -U flash-attn --no-build-isolation

INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 165.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 226.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 78.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.56.0
    Uninstalling transformers-4.56.0:
      Successfully uninstalled transformers-4.56.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.10.1
    Uninstalling accelerate-1.10.1:
      Successfully uninstalled accelerate-1.10.1

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgr

In [2]:
sh = """
HERE=$(pwd)

now=`date '+%Y%m%d_%H%M%S'`
JOBNAME=sqa_Qwen_LoRA_${now}
LOGDIR=$HERE/logs/$JOBNAME

export WANDB_API_KEY=73f8ff40bb7f8589e9bd1f476196a896f662cdfa # sqa's wandb key

mkdir -p ${LOGDIR}
# sudo chmod 777 -R ${LOGDIR}
echo 'Log dir: '$LOGDIR

echo 'login wandb'
python -m wandb login $WANDB_API_KEY
sleep 1
python -m wandb login
"""

with open('script.sh', 'w') as file:
  file.write(sh)

!bash script.sh

Log dir: /root/logs/sqa_Qwen_LoRA_20251202_031527
login wandb
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: Currently logged in as: sqa24 (zhh24-massachusetts-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
import re
import os
import yaml
import wandb
import torch
import torch.nn as nn
import torch.nn.functional as F
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

BASE_CONFIG = {
    "model": {
        "name": "Qwen/Qwen3-8B",
        # "name": "Qwen/Qwen3-0.6B",
        "torch_dtype": "bfloat16",
        "attn_implementation": "flash_attention_2",
        "cache_dir": ".cache/models",
        "device": "cuda",
    },
    "lora": {
        "r": 32,
        "lora_alpha": 64,
        "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "task_type": "CAUSAL_LM",
    },
    "training": {
        "max_steps": 500,
        "learning_rate": 1e-5,
        "gradient_accumulation_steps": 2,
        "logging_steps": 10,
        "bf16": True,
        "fp16": False,
        "max_prompt_length": 512,
        "max_completion_length": 1024,
        "do_sample": True,
        "temperature": 0.8,
        "top_p": 0.95,
    },
    "dataset": {
        "name": "openai/gsm8k",
        "config": "main",
        "split": "train",
        "cache_dir": ".cache/datasets",
    },
    "reward": {
        "format_reward": 0.5,
        "correctness_reward": 1.5,
    },
    "wandb": {
        "project": "rl-qwen-gsm8k",
        "enabled": True,
    },
}

# ============================================================================
# System Prompt
# ============================================================================

SYSTEM_PROMPT = """Solve the math problem step by step. Put your reasoning inside <think>...</think> tags and your final numerical answer inside <answer>...</answer> tags."""


# ============================================================================
# Data Formatting
# ============================================================================

def format_example_gsm8k(ex):
    prompt = (
        "Solve the math problem step by step. Put your reasoning inside <think>...</think> tags and your final numerical answer inside\n"
        "\"#### <answer>...</answer>\".\n\n"
        f"Question:\n{ex['question']}\n\nAnswer:\n"
    )
    # GSM8K answer already ends with '#### <ans>'
    target = ex["answer"]
    full_text = prompt + target
    return {"text": full_text}

def format_example_math500(ex):
    prompt = (
        "Solve the math problem step by step. Put your reasoning inside <think>...</think> tags and your final numerical answer inside\n"
        "\"#### <answer>...</answer>\".\n\n"
        f"Question:\n{ex['problem']}\n\nSolution:\n"
    )
    solution = ex["solution"]
    answer = ex["answer"]
    full_text = prompt + solution + f"\n#### {answer}"
    return {"text": full_text}

def check_format(text: str) -> bool:
    if "#### " in text:
        return True
    return False

def extract_answer(text: str) -> str:
    """Extract answer from response."""
    answer = text.split("####")[-1].strip()
    return answer


# ============================================================================
# Config Utilities
# ============================================================================

def deep_merge(base: dict, override: dict) -> dict:
    """
    Deep merge two dictionaries. Override values take precedence.
    Nested dicts are merged recursively, other values are replaced.
    """
    result = base.copy()
    for key, value in override.items():
        if key in result and isinstance(result[key], dict) and isinstance(value, dict):
            result[key] = deep_merge(result[key], value)
        else:
            result[key] = value
    return result


def load_config(config_path: str, overrides: dict = None) -> dict:
    """
    Load YAML config with inheritance support.

    If the config contains a 'base' key, it will first load the base config
    and then merge the current config on top of it.

    Example:
        # configs/grpo.yaml
        base: base.yaml
        method: grpo
        training:
          output_dir: outputs/grpo
    """
    import os

    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    # Handle inheritance from base config
    if "base" in config:
        base_path = config.pop("base")
        # Resolve relative path from the config file's directory
        config_dir = os.path.dirname(config_path)
        base_full_path = os.path.join(config_dir, base_path)

        # Load base config (recursively, to support multi-level inheritance)
        base_config = load_config(base_full_path)

        # Merge: base config + current config
        config = deep_merge(base_config, config)

    # Apply CLI overrides last (highest priority)
    if overrides:
        for key, value in overrides.items():
            keys = key.split(".")
            d = config
            for k in keys[:-1]:
                d = d.setdefault(k, {})
            try:
                d[keys[-1]] = yaml.safe_load(value)
            except:
                d[keys[-1]] = value

    return config


def get_torch_dtype(dtype_str: str):
    """Convert string to torch dtype."""
    mapping = {
        "bfloat16": torch.bfloat16,
        "float16": torch.float16,
        "float32": torch.float32,
    }
    return mapping.get(dtype_str, torch.bfloat16)

# ============================================================================
# End of Helpers
# ============================================================================


def train_sft(config: dict) -> str:
    """
    Train using SFT on GSM8K dataset with LoRA adaptation.

    Args:
        config: Configuration dictionary with model, lora, training, dataset, reward, wandb settings.

    Returns:
        Path to the saved model.
    """
    model_config = config["model"]
    lora_config = config["lora"]
    training_config = config["training"]
    dataset_config = config["dataset"]

    if config["wandb"]["enabled"]:
        os.environ["WANDB_PROJECT"] = config["wandb"]["project"]
        wandb.login()

    # Add timestamp to output_dir
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    output_dir = f"{training_config['output_dir']}-{timestamp}"

    # Model
    tokenizer = AutoTokenizer.from_pretrained(
        model_config["name"],
        cache_dir=model_config["cache_dir"],
    )
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_config["name"],
        dtype=get_torch_dtype(model_config["torch_dtype"]),
        attn_implementation=model_config["attn_implementation"],
        cache_dir=model_config["cache_dir"],
        device_map="cuda:0",
    )

    # Dataset
    dataset = load_dataset(
        dataset_config["name"],
        dataset_config["config"],
        split=dataset_config["split"],
        cache_dir=dataset_config["cache_dir"],
    )
    if dataset_config["name"] == "openai/gsm8k":
        dataset = dataset.map(format_example_gsm8k)
    elif dataset_config["name"] == "HuggingFaceH4/MATH-500":
        dataset = dataset.map(format_example_math500)
    else:
        raise ValueError(f"Unsupported dataset: {dataset_config['name']}")

    def tokenize_fn(ex):
        out = tokenizer(
            ex["text"],
            truncation=True,
            max_length=training_config["max_prompt_length"] + training_config["max_completion_length"],
            padding="max_length",
        )
        out["labels"] = out["input_ids"].copy()
        return out

    tokenized_dataset = dataset.map(
        tokenize_fn,
        batched=True,
        remove_columns=dataset.column_names,
    )

    # Training args
    training_args = TrainingArguments(
        output_dir=output_dir,
        max_steps=training_config["max_steps"],
        per_device_train_batch_size=training_config["per_device_train_batch_size"],
        gradient_accumulation_steps=training_config["gradient_accumulation_steps"],
        num_train_epochs=training_config["num_train_epochs"],
        learning_rate=training_config["learning_rate"],
        logging_steps=training_config["logging_steps"],
        bf16=training_config["bf16"],
        fp16=training_config["fp16"],
        # save_strategy="steps",
        # save_steps=training_config.get("save_steps", 100),
        # eval_strategy="steps",
        # eval_steps=training_config.get("eval_steps", 100),
        report_to="wandb" if config["wandb"]["enabled"] else "none",
        disable_tqdm=False,
        run_name=f"sft-{Path(output_dir).name}",
        # temperature=training_config["temperature"],
        # top_p=training_config["top_p"],
    )

    peft_config = LoraConfig(
        r=lora_config["r"],
        lora_alpha=lora_config["lora_alpha"],
        target_modules=lora_config["target_modules"],
        task_type=lora_config["task_type"],
    )

    trainer = Trainer(
        model=get_peft_model(model, peft_config),
        args=training_args,
        train_dataset=tokenized_dataset,
    )

    # Train
    print(f"\n{'='*60}")
    print(f"Training with SFT on GSM8K")
    print(f"Output: {output_dir}")
    print(f"{'='*60}\n")

    trainer.train()

    # Save
    final_path = f"{output_dir}-final"
    trainer.save_model(final_path)
    print(f"\nModel saved to {final_path}")

    return final_path

def main(config: dict) -> str:
    final_path = train_sft(config)
    return final_path

if __name__ == "__main__":
    _SFT_OVERRIDES_gsm8k = {
        "training": {
            "output_dir": "outputs/qwen-sft-gsm8k",
            "per_device_train_batch_size": 4,
            "num_train_epochs": 4,
        }
    }
    _SFT_OVERRIDES_math500 = {
        "training": {
            "output_dir": "outputs/qwen-sft-math500",
            "per_device_train_batch_size": 4,
            "num_train_epochs": 4,
        },
        "dataset": {
            "name": "HuggingFaceH4/MATH-500",
            "config": None,
            "split": "test",
            "cache_dir": "./cache",
        }
    }
    CONFIG_gsm8k = deep_merge(BASE_CONFIG, _SFT_OVERRIDES_gsm8k)
    CONFIG_math500 = deep_merge(BASE_CONFIG, _SFT_OVERRIDES_math500)
    
    final_path_gsm8k = main(CONFIG_gsm8k)
    print(f"Training completed. Model saved at: {final_path_gsm8k}")
    
    final_path_math500 = main(CONFIG_math500)
    print(f"Training completed. Model saved at: {final_path_math500}")

wandb: Currently logged in as: sqa24 (zhh24-massachusetts-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...


Training with SFT on GSM8K
Output: outputs/qwen-sft-gsm8k-20251202-031539



wandb: setting up run 3qocvxmv
wandb: Tracking run with wandb version 0.23.0
wandb: Run data is saved locally in /root/wandb/run-20251202_031647-3qocvxmv
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run sft-qwen-sft-gsm8k-20251202-031539
wandb: ⭐️ View project at https://wandb.ai/zhh24-massachusetts-institute-of-technology/rl-qwen-gsm8k
wandb: 🚀 View run at https://wandb.ai/zhh24-massachusetts-institute-of-technology/rl-qwen-gsm8k/runs/3qocvxmv


Step,Training Loss
10,7.857900
20,1.519500
30,0.622700
40,0.500100
50,0.386300
60,0.326100
70,0.294600
80,0.250200
90,0.241700
100,0.238600


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]


Model saved to outputs/qwen-sft-gsm8k-20251202-031539-final
Training completed. Model saved at: outputs/qwen-sft-gsm8k-20251202-031539-final


In [ ]:
# Evaluation

def evaluate_model(
    model_path: str,
    config: dict,
    num_samples: int = None,
    batch_size: int = 8,
    eval_base_model: bool = False,
):
    """
    Evaluate the trained model on GSM8K test set.
    
    Args:
        model_path: Path to the saved LoRA model.
        config: Configuration dictionary.
        num_samples: Number of samples to evaluate (None for all).
        batch_size: Batch size for generation.
        eval_base_model: Whether to evaluate the base model.
    
    Returns:
        Dictionary with evaluation metrics.
    """
    from peft import PeftModel
    
    model_config = config["model"]
    training_config = config["training"]
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        model_config["name"],
        cache_dir=model_config["cache_dir"],
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"  # For batch generation
    
    # Load base model
    base_model = AutoModelForCausalLM.from_pretrained(
        model_config["name"],
        dtype=get_torch_dtype(model_config["torch_dtype"]),
        attn_implementation=model_config["attn_implementation"],
        cache_dir=model_config["cache_dir"],
        device_map="cuda:0",
    )

    if eval_base_model:
        base_model.eval()
    
    # Load LoRA adapters
    model = PeftModel.from_pretrained(base_model, model_path)
    model.eval()
    
    # Load test dataset
    dataset_config = config["dataset"]
    test_dataset = load_dataset(
        dataset_config["name"],
        dataset_config["config"],
        split="test",
        cache_dir=dataset_config["cache_dir"],
    )
    
    if num_samples is not None:
        test_dataset = test_dataset.select(range(min(num_samples, len(test_dataset))))
    
    print(f"Evaluating on {len(test_dataset)} samples...")
    
    def _evaluate(model):
        # Metrics
        correct = 0
        format_correct = 0
        total = 0
        results = []
        
        # Process in batches
        for i in tqdm(range(0, len(test_dataset), batch_size), desc="Evaluating"):
            batch = test_dataset[i:i + batch_size]
            questions = batch["question"]
            answers = batch["answer"]
            
            # Format prompts
            prompts = []
            for q in questions:
                prompt = (
                    "Solve the math problem step by step. Put your reasoning inside <think>...</think> tags and your final numerical answer inside\n"
                    "\"#### <answer>...</answer>\".\n\n"
                    f"Question:\n{q}\n\nAnswer:\n"
                )
                prompts.append(prompt)
            
            # Tokenize
            inputs = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=training_config["max_prompt_length"],
            ).to(model.device)
            
            # Generate
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=training_config["max_completion_length"],
                    do_sample=training_config["do_sample"],
                    temperature=training_config["temperature"],
                    top_p=training_config["top_p"],
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            
            # Decode responses
            responses = tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            
            # Evaluate each response
            for prompt, response, answer in zip(prompts, responses, answers):
                total += 1
                
                # Check format
                if check_format(response):
                    format_correct += 1
                
                # Extract and compare answer
                extracted = extract_answer(response)
                correct_val = answer.split("#### ")[-1].strip()
                
                is_correct = extracted == correct_val
                if is_correct:
                    correct += 1
                
                results.append({
                    "question": prompt,
                    "response": response,
                    "extracted_answer": extracted,
                    "correct_answer": correct_val,
                    "is_correct": is_correct,
                    "has_correct_format": check_format(response),
                })
        
        # Compute metrics
        accuracy = correct / total if total > 0 else 0
        format_accuracy = format_correct / total if total > 0 else 0
        
        metrics = {
            "accuracy": accuracy,
            "format_accuracy": format_accuracy,
            "correct": correct,
            "format_correct": format_correct,
            "total": total,
        }
        
        print(f"\n{'='*60}")
        print(f"Evaluation Results")
        print(f"{'='*60}")
        print(f"Total samples: {total}")
        print(f"Correct answers: {correct} ({accuracy*100:.2f}%)")
        print(f"Correct format: {format_correct} ({format_accuracy*100:.2f}%)")
        print(f"{'='*60}\n")
        
        return metrics, results

    if eval_base_model:
        print("\nEvaluating base model...")
        base_metrics, base_results = _evaluate(base_model)
    
    print("\nEvaluating trained model...")
    lora_metrics, lora_results = _evaluate(model)
    
    return lora_metrics, lora_results if not eval_base_model else (
        base_metrics, base_results, lora_metrics, lora_results
    )

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Evaluating on 1319 samples...


Evaluating: 100%|██████████████████████████████████████████████████████████████| 42/42 [19:31<00:00, 27.90s/it]


Evaluation Results
Total samples: 1319
Correct answers: 1012 (76.72%)
Correct format: 1318 (99.92%)


--- Sample Results ---

Example 1:
  Extracted: 10
  Correct:   18
  Match:     False
  Format OK: True

Example 2:
  Extracted: 3
  Correct:   3
  Match:     True
  Format OK: True

Example 3:
  Extracted: 70000
  Correct:   70000
  Match:     True
  Format OK: True



In [ ]:
# Run evaluation for gsm8k
base_metrics_gsm8k, base_results_gsm8k, lora_metrics_gsm8k, lora_results_gsm8k = evaluate_model(
    model_path=final_path_gsm8k,
    config=CONFIG_gsm8k,
    num_samples=None,  # Evaluate on 100 samples for quick testing; set to None for full evaluation
    batch_size=32,
    eval_base_model=True,
)

# Show a few example results
print("\n--- Sample Results for base model on GSM8K ---\n")
for i, result in enumerate(base_results_gsm8k[:3]):
    print(f"Example {i+1}:")
    print(f"  Question:  {result["question"]}")
    print(f"  Response:  {result["response"]}")
    print(f"  Extracted: {result['extracted_answer']}")
    print(f"  Correct:   {result['correct_answer']}")
    print(f"  Match:     {result['is_correct']}")
    print(f"  Format OK: {result['has_correct_format']}")
    print()

print("\n--- Sample Results for trained model on GSM8K ---\n")
for i, result in enumerate(lora_results_gsm8k[:3]):
    print(f"Example {i+1}:")
    print(f"  Question:  {result["question"]}")
    print(f"  Response:  {result["response"]}")
    print(f"  Extracted: {result['extracted_answer']}")
    print(f"  Correct:   {result['correct_answer']}")
    print(f"  Match:     {result['is_correct']}")
    print(f"  Format OK: {result['has_correct_format']}")
    print()

In [ ]:
# Run evaluation for math500
base_metrics_math500, base_results_math500, lora_metrics_math500, lora_results_math500 = evaluate_model(
    model_path=final_path_math500,
    config=CONFIG_math500,
    num_samples=None,  # Evaluate on 100 samples for quick testing; set to None for full evaluation
    batch_size=32,
    eval_base_model=True,
)

# Show a few example results
print("\n--- Sample Results for base model on Math500 ---\n")
for i, result in enumerate(base_results_math500[:3]):
    print(f"Example {i+1}:")
    print(f"  Extracted: {result['extracted_answer']}")
    print(f"  Correct:   {result['correct_answer']}")
    print(f"  Match:     {result['is_correct']}")
    print(f"  Format OK: {result['has_correct_format']}")
    print()

print("\n--- Sample Results for trained model on Math500 ---\n")
for i, result in enumerate(lora_results_math500[:3]):
    print(f"Example {i+1}:")
    print(f"  Extracted: {result['extracted_answer']}")
    print(f"  Correct:   {result['correct_answer']}")
    print(f"  Match:     {result['is_correct']}")
    print(f"  Format OK: {result['has_correct_format']}")
    print()


--- Sample Results ---

Example 1:
  Question:  Solve the math problem step by step. Put your reasoning inside <think>...</think> tags and your final numerical answer inside
"#### <answer>...</answer>".

Question:
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Answer:

  Response:  She bakes muffins that each take 4 eggs every day, so that's 4*2=<<4*2=8>>8 eggs.
She eats 3 duck eggs for breakfast every day, so that's 3+8=<<3+8=11>>11 eggs.
That means she can only sell 16-11=<<16-11=5>>5 eggs daily at the market.
So, she makes 5*2=$<<5*2=10>>10.
#### 10
  Extracted: 10
  Correct:   18
  Match:     False
  Format OK: True

Example 2:
  Question:  Solve the math problem step by step. Put your reasoning inside <think>...</think> tags and your final numerical